# CE497 · Water-column collapse: SPH ground truth vs a trained GNN

**Run in Colab:** upload → **Runtime → Run all**. A T4 GPU is optional; CPU also works.

This compact notebook generates many **SPH water-column collapses**, trains a small **message-passing GNN** from those data, and then compares SPH and GNN on a new water column.

You will see:
1. synchronized SPH vs GNN water animations;
2. a second animation showing the GNN's **nodes and changing edges**.

The intended classroom runtime is **< 5 min** on a typical Colab runtime. Runtime safeguards automatically reduce the amount of training if a shared runtime is slow.

> This is a deliberately small **physics-informed teaching model**, not the full DeepMind GNS and not validated engineering SPH software. The GNN learns the local pair-interaction strength; graph construction, gravity, kernel-gradient direction, walls, and time integration remain explicit.

## 1 · SPH: High-Level Physics

**Smoothed Particle Hydrodynamics (SPH)** represents water using moving particles rather than a fixed mesh. Each particle \(i\) interacts only with nearby particles inside a smoothing radius \(h\).

### Density

The density at particle \(i\) is estimated from its neighboring particles:

$$
\rho_i = \sum_j m_j W\left(|\mathbf{x}_i-\mathbf{x}_j|,h\right)
$$

where:

- \(\rho_i\) is the density of particle \(i\),
- \(m_j\) is the mass of neighboring particle \(j\),
- \(W\) is the **smoothing kernel**,
- \(h\) is the smoothing radius.

The kernel gives larger weight to nearby particles and smaller weight to particles farther away.

---

### Momentum Equation

The acceleration of each particle is obtained from the forces exerted by its neighbors:

$$
\frac{d\mathbf{v}_i}{dt}
=
-\sum_j m_j
\left(
\frac{P_i}{\rho_i^2}
+
\frac{P_j}{\rho_j^2}
+
\Pi_{ij}
\right)
\nabla W_{ij}
+
\mathbf{g}
$$

where:

- \(\mathbf{v}_i\) is the velocity of particle \(i\),
- \(P_i\) and \(P_j\) are particle pressures,
- \(\Pi_{ij}\) represents artificial viscosity,
- \(\nabla W_{ij}\) gives the direction and strength of the particle interaction,
- \(\mathbf{g}\) is gravitational acceleration.

In simple terms:

> **Neighboring particles push and resist one another, while gravity pulls all particles downward.**

---

### Pressure

This notebook uses a simple weakly-compressible pressure law:

$$
P_i
=
c_0^2
\max\left(\rho_i-\rho_0,\,0\right)
$$

with

$$
\rho_0 = 1
$$

where \(c_0\) is the numerical speed of sound and \(\rho_0\) is the reference density.

If particles become crowded,

$$
\rho_i > \rho_0,
$$

pressure increases and pushes the particles apart.

---

### SPH Solver Logic

At every time step, the solver performs approximately the following operations:

```text
Particle positions + velocities
            ↓
Find neighboring particles
            ↓
Estimate density ρ
            ↓
Calculate pressure P
            ↓
Calculate pressure + viscous forces
            ↓
Sum forces → acceleration
            ↓
Update velocity
            ↓
Update particle position
            ↓
Repeat

In [1]:
#@title 1 · Define the SPH water-column solver
import time, sys, subprocess, importlib.util
START = time.perf_counter()
missing = [p for p in ('numpy', 'scipy', 'numba', 'torch')
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import math, json, uuid
import numpy as np
import torch
from torch import nn
from numba import njit
from scipy.spatial import cKDTree
from IPython.display import HTML, display
np.random.seed(12); torch.manual_seed(12); torch.set_num_threads(2)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training device: {DEVICE}. SPH and the small rollout graph run on CPU.')
# 2-D weakly-compressible SPH. Reference density rho0 = 1 in normalized units.
# Each time step follows: neighbors -> density -> pair forces -> acceleration -> move.
# Wendland C2 kernel, linear positive pressure, Monaghan artificial viscosity.
D=.035; H=2.6*D; MASS=D*D; C0=10.; ALPHA=.15
DT=np.float32(.001); STEPS=1000; SAVE=8
BOX=np.array([1.2,.95],np.float32); PAD=.5*D
# Three layers of stationary ghost particles support pressure near the tank walls.
b=[]
for l in range(3):
 for x in np.arange(-2*D,1.2+3*D,D):b.append([x,-(l+.5)*D]);b.append([x,.95+(l+.5)*D])
 for y in np.arange(.5*D,.95,D):b.append([-(l+.5)*D,y]);b.append([1.2+(l+.5)*D,y])
WALL=np.array(b,np.float32)
@njit
def density(x):
 n=len(x);r=np.zeros(n,np.float32); K=7/(math.pi*H*H)
 for i in range(n):
  for j in range(n+len(WALL)):
   rx=x[i,0]-(x[j,0] if j<n else WALL[j-n,0]);ry=x[i,1]-(x[j,1] if j<n else WALL[j-n,1]);r2=rx*rx+ry*ry
   if r2<H*H:
    q=math.sqrt(r2)/H;r[i]+=MASS*K*(1-q)**4*(1+4*q)
 return r
@njit
def force(x,v,rho):
 n=len(x);a=np.zeros_like(x)
 for i in range(n):
  a[i,1]=-9.81
  for j in range(n+len(WALL)):
   if j==i:continue
   rx=x[i,0]-(x[j,0] if j<n else WALL[j-n,0]);ry=x[i,1]-(x[j,1] if j<n else WALL[j-n,1]);r2=rx*rx+ry*ry
   if 1.e-12<r2<H*H:
    q=math.sqrt(r2)/H;ri=rho[i];rj=rho[j] if j<n else ri
    ux=v[i,0]-(v[j,0] if j<n else 0.);uy=v[i,1]-(v[j,1] if j<n else 0.)
    mu=H*min(ux*rx+uy*ry,0.)/(r2+.01*H*H)
    coeff=C0*C0*(max(ri-1.,0.)/(ri*ri)+max(rj-1.,0.)/(rj*rj))-ALPHA*C0*mu/(.5*(ri+rj))
    grad=140/(math.pi*H**4)*(1-q)**3
    a[i,0]+=MASS*coeff*grad*rx;a[i,1]+=MASS*coeff*grad*ry
 return a
# The same wall-contact rule and explicit integrator are later used by SPH and GNN.
# The hard contact is only a leakage safeguard; tangential motion remains free.
@njit
def move(x,v,a):
 v=v+DT*a;x=x+DT*v
 for i in range(len(x)):
  for k in range(2):
   if x[i,k]<.1*D:x[i,k]=.1*D;v[i,k]=max(0.,-.1*v[i,k])
   if x[i,k]>BOX[k]-.1*D:x[i,k]=BOX[k]-.1*D;v[i,k]=min(0.,-.1*v[i,k])
 return x,v
@njit
def simulate(x):
 v=np.zeros_like(x);path=np.empty((STEPS//SAVE+1,len(x),2),np.float32);vel=np.empty_like(path);path[0]=x;vel[0]=v
 for s in range(STEPS):
  r=density(x);a=force(x,v,r);x,v=move(x,v,a)
  if (s+1)%SAVE==0:path[(s+1)//SAVE]=x;vel[(s+1)//SAVE]=v
 return path,vel

# Create slightly different water columns so training does not see one identical geometry.
def column(seed,test=False):
 rng=np.random.default_rng(seed);nx=10 if test else int(rng.integers(8,13));ny=20 if test else int(rng.integers(17,24))
 x=np.stack(np.meshgrid(np.arange(nx)*D,np.arange(ny)*D),-1).reshape(-1,2).astype(np.float32)
 x+=np.array([PAD,PAD],np.float32);x+=rng.uniform(-.015*D,.015*D,x.shape).astype(np.float32)
 return x



Training device: cuda. SPH and the small rollout graph run on CPU.


## 2 · From Particles to a GNN

At each time step, we represent the fluid as a **dynamic radius graph**:

- **Node** = one water particle
- **Edge** = a connection between two particles whose distance is smaller than the SPH interaction radius \(h\)
- The graph changes automatically as particles move

The particle system

$$
\left\{
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_N
\right\}
$$

becomes a graph

$$
G=(V,E),
$$

where \(V\) contains the particle nodes and \(E\) contains the local particle-particle connections.

---

### From the SPH equation to a GNN

Recall that in SPH, the acceleration of particle \(i\) is obtained by summing the contributions from all neighboring particles:

$$
\mathbf{a}_i
=
\sum_{j\in\mathcal{N}(i)}
\mathbf{a}_{ij}
+
\mathbf{g}.
$$

In the SPH solver used in this notebook, the contribution from one neighboring particle \(j\) can be written schematically as

$$
\boxed{
\mathbf{a}_{ij}
=
C_{ij}\,\mathbf{b}_{ij}
}
$$

where:

- \(C_{ij}\) is a **scalar SPH pair-force coefficient**
- \(\mathbf{b}_{ij}\) contains the known geometric direction and kernel-gradient term

Therefore,

$$
\boxed{
\mathbf{a}_i
=
\sum_{j\in\mathcal{N}(i)}
C_{ij}\mathbf{b}_{ij}
+
\mathbf{g}
}
$$

This decomposition is important because, in this notebook, the GNN learns **only \(C_{ij}\)**.

The geometry of the particle interaction, gravity, and time integration remain explicitly known.

---

### What is the SPH pair-force coefficient?

For the simplified weakly-compressible SPH model used here,

$$
C_{ij}
=
C_{\mathrm{pressure}}
+
C_{\mathrm{viscosity}}.
$$

More specifically,

$$
\boxed{
C_{ij}
=
c_0^2
\left[
\frac{\max(\rho_i-\rho_0,0)}{\rho_i^2}
+
\frac{\max(\rho_j-\rho_0,0)}{\rho_j^2}
\right]
-
\frac{
\alpha c_0 \mu_{ij}
}{
\frac{1}{2}(\rho_i+\rho_j)
}
}
$$

where:

- \(\rho_i,\rho_j\) are the local particle densities
- \(\rho_0\) is the reference density
- \(c_0\) is the numerical speed of sound
- \(\alpha\) controls artificial viscosity
- \(\mu_{ij}\) describes how strongly particles \(i\) and \(j\) are approaching each other

The first part,

$$
c_0^2
\left[
\frac{\max(\rho_i-\rho_0,0)}{\rho_i^2}
+
\frac{\max(\rho_j-\rho_0,0)}{\rho_j^2}
\right],
$$

represents the **pressure interaction**.

The second part,

$$
-
\frac{
\alpha c_0 \mu_{ij}
}{
\frac{1}{2}(\rho_i+\rho_j)
},
$$

represents the **artificial-viscosity interaction**.

So \(C_{ij}\) is not a new physical quantity introduced by the GNN.

It is simply a convenient way of grouping terms that already exist inside the SPH momentum equation.

---

### The geometric part of the SPH interaction

The remaining vector term is

$$
\mathbf{b}_{ij}
=
m_j
K(q_{ij})
\left(
\mathbf{x}_i-\mathbf{x}_j
\right),
$$

where

$$
q_{ij}
=
\frac{
|\mathbf{x}_i-\mathbf{x}_j|
}{h}.
$$

For the kernel used in this notebook,

$$
K(q)
=
\frac{140}{\pi h^4}
(1-q)^3,
\qquad
q<1.
$$

Therefore,

$$
\mathbf{b}_{ij}
$$

contains:

- the direction from particle \(j\) to particle \(i\)
- the distance between the particles
- the SPH kernel-gradient dependence

The GNN does **not** need to learn this geometry.

---

### What information does the GNN see?

For every connected particle pair \(i\) and \(j\), we construct an edge feature vector

$$
\mathbf{z}_{ij}.
$$

In this notebook, the four edge features contain information about:

1. the smaller local density
2. the larger local density
3. the relative approach velocity
4. the normalized particle separation

Schematically,

$$
\boxed{
\mathbf{z}_{ij}
=
\left[
\rho_i,\,
\rho_j,\,
\mu_{ij},\,
q_{ij}
\right]
}
$$

with some simple numerical scaling applied in the code.

These are exactly the types of local quantities that determine the SPH pair interaction.

---

### What does the GNN learn?

Instead of directly calculating \(C_{ij}\) from the SPH equation, we ask a neural network to approximate it:

$$
\boxed{
\hat{C}_{ij}
=
f_{\theta}
\left(
\mathbf{z}_{ij}
\right)
}
$$

where:

- \(\mathbf{z}_{ij}\) = local edge features
- \(f_{\theta}\) = neural network
- \(\theta\) = trainable weights and biases
- \(\hat{C}_{ij}\) = GNN prediction of the SPH pair-force coefficient

For numerical convenience, the actual network is trained on

$$
Y_{ij}
=
\frac{C_{ij}}{10},
$$

so the network predicts

$$
\boxed{
\hat{Y}_{ij}
=
f_{\theta}(\mathbf{z}_{ij})
\approx
\frac{C_{ij}}{10}
}
$$

and during the rollout we recover

$$
\hat{C}_{ij}
=
10\hat{Y}_{ij}.
$$

---

### Neural-network architecture

The shared edge neural network has the architecture

$$
\boxed{
4
\rightarrow
48
\rightarrow
48
\rightarrow
1
}
$$

or

$$
\text{4 edge features}
\rightarrow
\text{48 hidden neurons}
\rightarrow
\text{48 hidden neurons}
\rightarrow
\text{1 SPH pair-force coefficient}.
$$

Only the **weights and biases** inside these neural-network layers are trained.

The network contains

$$
\boxed{
2641\ \text{trainable parameters}
}
$$

and the **same neural-network weights are reused for every edge** in the fluid.

---

### Message passing

Once the GNN predicts the coefficient for edge \(i-j\),

$$
\hat{C}_{ij}
=
10f_{\theta}(\mathbf{z}_{ij}),
$$

we reconstruct the particle-pair acceleration message:

$$
\boxed{
\mathbf{m}_{ij}
=
\hat{C}_{ij}
\mathbf{b}_{ij}
}
$$

Ideally,

$$
\mathbf{m}_{ij}
\approx
C_{ij}\mathbf{b}_{ij}
=
\mathbf{a}_{ij}^{\mathrm{SPH}}.
$$

All messages arriving at particle \(i\) are then summed:

$$
\boxed{
\mathbf{a}_i^{\mathrm{GNN}}
=
\sum_{j\in\mathcal{N}(i)}
\mathbf{m}_{ij}
+
\mathbf{g}
}
$$

The GNN therefore replaces this SPH calculation:

$$
\boxed{
C_{ij}^{\mathrm{SPH}}
\quad\longrightarrow\quad
\hat{C}_{ij}^{\mathrm{GNN}}
}
$$

while keeping the rest of the particle mechanics explicit.

---

### What is minimized during training?

For every sampled SPH particle pair, we know the target

$$
Y_{ij}
=
\frac{C_{ij}^{\mathrm{SPH}}}{10}.
$$

The network predicts

$$
\hat{Y}_{ij}
=
f_{\theta}(\mathbf{z}_{ij}).
$$

The notebook minimizes the weighted mean-square error

$$
\boxed{
\mathcal{L}
=
\frac{1}{N}
\sum_{ij}
\frac{
\left(
\hat{Y}_{ij}
-
Y_{ij}
\right)^2
}{
0.15+Y_{ij}
}
}
$$

or equivalently,

$$
\boxed{
\theta^{*}
=
\arg\min_{\theta}
\mathcal{L}.
}
$$

The numerator

$$
\left(
\hat{Y}_{ij}
-
Y_{ij}
\right)^2
$$

penalizes incorrect GNN predictions.

The denominator

$$
0.15+Y_{ij}
$$

prevents a small number of very large pressure interactions from dominating the training, so smaller and more common interactions also contribute to the loss.

---

### Training flow

```text
SPH simulations
      ↓
particle positions + velocities + densities
      ↓
build radius graph
      ↓
calculate local edge features z_ij
      ↓
SPH equation calculates target C_ij
      ↓
GNN predicts Ĉ_ij
      ↓
compare GNN prediction with SPH target
      ↓
calculate loss
      ↓
backpropagation
      ↓
Adam updates GNN weights and biases

In [2]:
#@title 2 · Define the dynamic graph and edge-message GNN
def graph(x, v):
    """Build a radius graph from the CURRENT particle state only.
    Nodes = particles; edges = pairs closer than H.
    No future SPH positions are used."""
    n = len(x)
    allx = np.concatenate((x, WALL))
    allv = np.concatenate((v, np.zeros_like(WALL)))
    pairs = cKDTree(allx).query_pairs(H, output_type='ndarray')
    pairs = pairs[pairs[:, 0] < n]  # omit ghost-ghost interactions
    i, j = pairs.T
    dr = allx[i] - allx[j]
    rr = (dr * dr).sum(1)
    q = np.sqrt(rr) / H
    rho = density(x)  # first, fixed geometric neighbor aggregation
    ri = rho[i]
    rj = np.where(j < n, rho[np.minimum(j, n-1)], ri)
    dv = allv[i] - allv[j]
    mu = H * np.minimum((dv * dr).sum(1), 0) / (rr + .01*H*H)
    # Four local edge features are given to the shared neural network.
    # The same MLP weights are reused for every edge and every time step.
    # Symmetric features help fluid-fluid messages remain equal and opposite.
    z = np.stack((20*(np.minimum(ri,rj)-1),
                  20*(np.maximum(ri,rj)-1), -mu, q), 1).astype(np.float32)
    basis = (MASS*140/(np.pi*H**4)*(1-q)**3)[:,None] * dr
    return z, i, j, basis, ri, rj, mu

class PairGNN(nn.Module):
    """Shared edge MLP + sum aggregation on a graph that changes as particles move."""
    def __init__(self):
        super().__init__()
        self.edge = nn.Sequential(nn.Linear(4,48), nn.SiLU(),
                                 nn.Linear(48,48), nn.SiLU(), nn.Linear(48,1))
    def forward(self, edge_features):
        return torch.nn.functional.softplus(self.edge(edge_features)).squeeze(-1)

@torch.inference_mode()
def learned_acceleration(model, x, v):
    z, i, j, basis, *_ = graph(x, v)
    messages = 10 * model(torch.from_numpy(z)).numpy()[:,None] * basis
    a = np.zeros_like(x); a[:,1] = -9.81  # known gravity
    np.add.at(a, i, messages)
    fluid = j < len(x)
    np.add.at(a, j[fluid], -messages[fluid])
    return a

# Only the Linear-layer weights and biases below are trained.
_preview = PairGNN()
n_params = sum(p.numel() for p in _preview.parameters() if p.requires_grad)
print(_preview)
print(f"Trainable GNN weights + biases: {n_params:,}")
del _preview


PairGNN(
  (edge): Sequential(
    (0): Linear(in_features=4, out_features=48, bias=True)
    (1): SiLU()
    (2): Linear(in_features=48, out_features=48, bias=True)
    (3): SiLU()
    (4): Linear(in_features=48, out_features=1, bias=True)
  )
)
Trainable GNN weights + biases: 2,641


## 3 · Generate training data with SPH

We now create many independent water-column collapses with slightly different widths, heights, and small initial particle offsets.

A complete trajectory contains many saved states, and each state contains many particle-pair edges. Therefore a few dozen trajectories already create **hundreds of thousands of local interaction examples**.

Each training example is:

$$
\text{local edge features } \mathbf{z}_{ij}
\;\longrightarrow\;
\text{SPH pair-interaction strength } c_{ij}
$$

In [3]:
#@title Generate up to 96 SPH column-collapse trajectories
# Generate independent SPH column collapses, not repeated copies of one movie.
# Every saved SPH state becomes a new local graph; sampled edges become training examples.
# Width, height, and small initial particle offsets change between trajectories.
N_TRAJECTORIES = 96
rng = np.random.default_rng(24)
Xs, Ys = [], []
data_start = time.perf_counter()
for seed in range(N_TRAJECTORIES):
    path, vel = simulate(column(seed))
    for frame in range(0, len(path), 2):
        z, _, _, _, ri, rj, mu = graph(path[frame], vel[frame])
        # Supervised target: the SPH pair-interaction coefficient for this edge.
        # The GNN will learn a mapping: local edge features -> interaction strength.
        # This formula is used ONLY for training, never in the GNN rollout.
        target = (C0*C0*(np.maximum(ri-1,0)/ri**2 + np.maximum(rj-1,0)/rj**2)
                  - ALPHA*C0*mu/(.5*(ri+rj))) / 10
        # Randomly subsample edges so we can use many trajectories while staying fast.
        keep = rng.choice(len(target), min(256,len(target)), replace=False)
        Xs.append(z[keep]); Ys.append(target[keep].astype(np.float32))
    # Runtime safeguard: finish the current trajectory before stopping.
    if time.perf_counter()-data_start > 95 or time.perf_counter()-START > 160:
        break
n_trajectories = seed + 1
X = torch.from_numpy(np.concatenate(Xs)).to(DEVICE)
Y = torch.from_numpy(np.concatenate(Ys)).to(DEVICE)
del Xs, Ys, path, vel
print(f'{n_trajectories} SPH trajectories; {n_trajectories*63:,} sampled states; '
      f'{len(Y):,} local interaction examples. '
      f'Data generation: {time.perf_counter()-data_start:.0f} s.')

94 SPH trajectories; 5,922 sampled states; 1,516,032 local interaction examples. Data generation: 96 s.


## 4 · Train the GNN

Training begins from random weights.

For every mini-batch:
1. sample local SPH edge examples;
2. let the edge MLP predict interaction strengths;
3. compare predictions with SPH targets;
4. backpropagate the loss;
5. use **Adam** to update the MLP's weights and biases.

The learned weights are fixed after training and reused throughout the later fluid rollout.

In [4]:
#@title 3 · Train the GNN from scratch
# Train from random weights. No pretrained checkpoint, trajectory replay, or download.
# Adam updates ONLY the weights/biases inside the shared edge MLP.
model = PairGNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=.002)
MAX_UPDATES = 6000
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, MAX_UPDATES, eta_min=.0002)
train_start = time.perf_counter()
for step in range(MAX_UPDATES):
    # Mini-batch = random local particle-pair examples collected from SPH.
    ids = torch.randint(len(X), (2048,), device=DEVICE)
    prediction = model(X[ids])
    # Compare the learned edge interaction with the SPH target.
    # This weighted MSE keeps small/common interactions from being ignored.
    loss = ((prediction-Y[ids]).square()/(.15+Y[ids])).mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()      # backpropagate gradients through the MLP
    optimizer.step()     # change trainable weights and biases
    scheduler.step()
    if (step+1) % 100 == 0:
        if DEVICE == 'cuda': torch.cuda.synchronize()
        if time.perf_counter()-train_start > 85 or time.perf_counter()-START > 245:
            break
model = model.cpu().eval()  # tiny dynamic graphs avoid per-step GPU transfers
print(f'Trained for {step+1:,} updates in {time.perf_counter()-train_start:.0f} s.')

Trained for 6,000 updates in 11 s.


## 5 · Held-out simulation: SPH vs GNN

The next water column was **not included in training**.

Both methods start from exactly the same initial particle coordinates.

- **Left:** SPH evolves from the explicit pressure/viscosity equations.
- **Right:** GNN evolves from its **own previous predictions**.

There is no reset or correction toward SPH during the GNN rollout.

For clear visualization in both light and dark Colab themes, the comparison uses **white water particles on a black background**.

In [5]:
#@title 5 · Run a held-out column: SPH ground truth vs GNN
# A held-out initialization inside the training family, NEVER used above.
# SPH and GNN start from exactly the same particle positions.
x0 = column(987, test=True)
truth, _ = simulate(x0.copy())
x, v = x0.copy(), np.zeros_like(x0)
predicted = [x.copy()]
for s in range(STEPS):
    # IMPORTANT: this uses the GNN's own CURRENT state, not the SPH future.
    a = learned_acceleration(model, x, v)
    x, v = move(x, v, a)
    if not np.isfinite(x).all():
        raise RuntimeError('The learned rollout became non-finite. Rerun training.')
    if (s+1) % SAVE == 0:
        predicted.append(x.copy())
predicted = np.stack(predicted)
# No resets, blends, or corrections toward SPH positions during prediction.
print(f'{len(x0)} water particles; 1,000 independent prediction steps. '
      f'Total compute so far: {time.perf_counter()-START:.0f} s.')

# Lightweight browser animation: no video encoder or slow Matplotlib rendering.
# Two synchronized canvases, a play/pause button, and a time slider.
uid = 'water_' + uuid.uuid4().hex[:10]
payload = json.dumps({'truth':truth.round(5).tolist(),
                      'gnn':predicted.round(5).tolist()}, separators=(',',':'))
html = r"""
<div id="__ID__" style="font-family:system-ui;max-width:1000px;padding:12px;background:#111;color:#fff;border-radius:10px">
  <div style="display:flex;gap:16px;flex-wrap:wrap">
    <div style="flex:1;min-width:280px"><h3>SPH ground truth</h3>
      <canvas data-name="truth" width="576" height="456" style="width:100%;background:#000;border:1px solid #666"></canvas></div>
    <div style="flex:1;min-width:280px"><h3>Trained GNN</h3>
      <canvas data-name="gnn" width="576" height="456" style="width:100%;background:#000;border:1px solid #666"></canvas></div>
  </div>
  <div style="display:flex;align-items:center;gap:12px;margin-top:12px">
    <button>Pause</button><input type="range" min="0" max="125" value="0" style="flex:1">
    <span style="min-width:100px"></span>
  </div>
  <p style="font-size:13px">Same initial water column. The right panel evolves from its own predictions; it does not read the left panel.</p>
</div>
<script>
(()=>{
 const root=document.getElementById('__ID__'), data=__DATA__;
 const canvases=[...root.querySelectorAll('canvas')];
 const slider=root.querySelector('input'), button=root.querySelector('button'), text=root.querySelector('span');
 let frame=0, playing=true, last=0;
 slider.max=data.truth.length-1;
 function draw(){
   canvases.forEach(c=>{
     const ctx=c.getContext('2d'), pts=data[c.dataset.name][frame];
     ctx.fillStyle='#000';ctx.fillRect(0,0,c.width,c.height);ctx.fillStyle='#fff';
     for(const p of pts){ctx.beginPath();ctx.arc(p[0]/1.2*c.width,(1-p[1]/.95)*c.height,2.8,0,2*Math.PI);ctx.fill();}
   });
   slider.value=frame; text.textContent='t = '+(frame*.008).toFixed(3);
 }
 button.onclick=()=>{playing=!playing;button.textContent=playing?'Pause':'Play';};
 slider.oninput=()=>{frame=+slider.value;playing=false;button.textContent='Play';draw();};
 function tick(now){
   if(!root.isConnected)return;
   if(playing && now-last>42){frame=(frame+1)%data.truth.length;draw();last=now;}
   requestAnimationFrame(tick);
 }
 draw();requestAnimationFrame(tick);
})();
</script>
""".replace('__ID__',uid).replace('__DATA__',payload)
display(HTML(html))

200 water particles; 1,000 independent prediction steps. Total compute so far: 127 s.


## 6 · Watch the GNN graph change

The next animation visualizes the **GNN prediction itself as a graph**:

- **white dots** = particle nodes;
- **cyan lines** = fluid–fluid edges within the real neighbor radius \(h\);
- **yellow node/edges** = one example node and its current neighbors.

As water moves, some neighbors separate and new neighbors approach, so the edge network continuously changes.

For readability, this picture omits ghost-wall nodes and wall edges. The actual GNN calculation above still includes the wall interactions.

In [6]:
#@title 6 · Animate GNN nodes and dynamic edges
uid2 = "graph_" + uuid.uuid4().hex[:10]
graph_payload = json.dumps(
    {"gnn": predicted.round(5).tolist(), "radius": float(H)},
    separators=(",", ":")
)

html2 = r"""
<div id="__ID__" style="font-family:system-ui;max-width:760px;padding:12px;background:#111;color:#fff;border-radius:10px">
  <h3>Dynamic GNN particle graph</h3>
  <canvas width="720" height="570" style="width:100%;background:#000;border:1px solid #666"></canvas>
  <div style="display:flex;align-items:center;gap:12px;margin-top:10px">
    <button>Pause</button><input type="range" value="0" style="flex:1"><span style="min-width:100px"></span>
  </div>
  <div style="font-size:13px;color:#ddd;margin-top:8px">
    <span style="color:white">●</span> node &nbsp;&nbsp;
    <span style="color:#36d7ff">━━</span> graph edge &nbsp;&nbsp;
    <span style="color:#ffd84d">●━━</span> highlighted neighborhood
  </div>
</div>
<script>
(()=>{
 const root=document.getElementById('__ID__'),data=__DATA__,c=root.querySelector('canvas'),ctx=c.getContext('2d');
 const slider=root.querySelector('input'),button=root.querySelector('button'),text=root.querySelector('span');
 let frame=0,playing=true,last=0;
 slider.min=0;slider.max=data.gnn.length-1;
 const hi=Math.floor(data.gnn[0].length/2), r2=data.radius*data.radius;
 const X=p=>p[0]/1.2*c.width, Y=p=>(1-p[1]/.95)*c.height;

 function draw(){
   const pts=data.gnn[frame];
   ctx.fillStyle='#000';ctx.fillRect(0,0,c.width,c.height);

   // Draw every true fluid-fluid radius edge.
   ctx.beginPath();
   for(let i=0;i<pts.length;i++)for(let j=i+1;j<pts.length;j++){
     const dx=pts[i][0]-pts[j][0],dy=pts[i][1]-pts[j][1];
     if(dx*dx+dy*dy<r2){ctx.moveTo(X(pts[i]),Y(pts[i]));ctx.lineTo(X(pts[j]),Y(pts[j]));}
   }
   ctx.strokeStyle='rgba(54,215,255,.24)';ctx.lineWidth=.8;ctx.stroke();

   // Highlight one node's current neighborhood.
   ctx.beginPath();
   for(let j=0;j<pts.length;j++)if(j!==hi){
     const dx=pts[hi][0]-pts[j][0],dy=pts[hi][1]-pts[j][1];
     if(dx*dx+dy*dy<r2){ctx.moveTo(X(pts[hi]),Y(pts[hi]));ctx.lineTo(X(pts[j]),Y(pts[j]));}
   }
   ctx.strokeStyle='#ffd84d';ctx.lineWidth=2;ctx.stroke();

   // Draw nodes on top of edges.
   ctx.fillStyle='#fff';
   for(const p of pts){ctx.beginPath();ctx.arc(X(p),Y(p),2.7,0,2*Math.PI);ctx.fill();}
   ctx.fillStyle='#ffd84d';ctx.beginPath();ctx.arc(X(pts[hi]),Y(pts[hi]),5,0,2*Math.PI);ctx.fill();

   slider.value=frame;text.textContent='t = '+(frame*.008).toFixed(3);
 }
 button.onclick=()=>{playing=!playing;button.textContent=playing?'Pause':'Play';};
 slider.oninput=()=>{frame=+slider.value;playing=false;button.textContent='Play';draw();};
 function tick(now){
   if(!root.isConnected)return;
   if(playing&&now-last>55){frame=(frame+1)%data.gnn.length;draw();last=now;}
   requestAnimationFrame(tick);
 }
 draw();requestAnimationFrame(tick);
})();
</script>
""".replace("__ID__",uid2).replace("__DATA__",graph_payload)

display(HTML(html2))


## What to notice

In the **SPH vs GNN** animation, compare the overall collapse, spreading front, splash near the far wall, and individual particle paths. The two simulations can look physically similar while gradually diverging because the GNN repeatedly feeds its own predictions back into the next step.

In the **graph animation**, notice an important distinction:

- the **trained neural-network weights stay fixed**;
- the **graph topology changes** as the particles move.


# References
*   DeepMind GNN website: https://sites.google.com/view/learning-to-simulate
*   DeepMind paper: https://cs.stanford.edu/people/jure/pubs/learning_to_simulate-icml20.pdf
*   DeepMind github repository: https://github.com/google-deepmind/deepmind-research/tree/master/learning_to_simulate

